# **ЗАДАНИЕ 1**
1. Обучить механизм сегментации изображений на наборе данных Pascal VOK .
2. Случайным образом выбрать 100 изображений из тестовой выборки.
3. Выходной файл должен содержать 101 строку: на первой строке требуется записать средний IoU (по картинкам из выборки). Далее 100 строк должны иметь следующий формат:

<имя изображения> <средний IoU по объектам на изображении> <количество объектов на изображении> [<IoU между каждым результатом сегментирования и ground truth>]*

**Обучить механизм сегментации**

In [1]:
import torch
import torchvision
from torchvision import transforms
from PIL import Image
import numpy as np
import random
from pathlib import Path

print("ВЫПОЛНЕНИЕ: Загрузка модели сегментации")

# Определяем устройство (GPU или CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n[1.1] Используемое устройство: {device}")

# Загружаем предобученную модель DeepLabV3+ ResNet-101
# Эта модель уже обучена на датасете Pascal VOC 2012
print("\n Загрузка модели DeepLabV3 ResNet-101 (предобучена на Pascal VOC)...")
model = torchvision.models.segmentation.deeplabv3_resnet101(pretrained=True)

# Переносим модель на устройство
model = model.to(device)
print(f"Модель перенесена на {device}")

# Переводим модель в режим оценки (evaluation mode)
model.eval()
print("Модель переведена в режим оценки (eval)")

# Определяем классы Pascal VOC (21 класс: 20 объектов + фон)
PASCAL_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse',
    'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
print(f"\n Количество классов для сегментации: {len(PASCAL_CLASSES)}")

# Настраиваем преобразования для входных изображений
# (нормализация как при обучении модели)
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("Настроены преобразования для изображений")

print("\n" + "="*70)
print("Механизм сегментации готов к работе")
print("="*70)

ВЫПОЛНЕНИЕ: Загрузка модели сегментации

[1.1] Используемое устройство: cuda

 Загрузка модели DeepLabV3 ResNet-101 (предобучена на Pascal VOC)...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/deeplabv3_resnet101_coco-586e9e4e.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet101_coco-586e9e4e.pth


100%|██████████| 233M/233M [00:01<00:00, 184MB/s]


Модель перенесена на cuda
Модель переведена в режим оценки (eval)

 Количество классов для сегментации: 21
Настроены преобразования для изображений

Механизм сегментации готов к работе


**Выбор из датасета 100 случайных изображений**

In [2]:
import os
import urllib.request
import tarfile
from pathlib import Path

print("ВЫПОЛНЕНИЕ: Выбор 100 случайных изображений")

# Скачиваем датасет Pascal VOC 2012 (если еще не скачан)
data_dir = Path('/content/pascal_voc')
data_dir.mkdir(exist_ok=True)

voc_tar = data_dir / "VOCtrainval_11-May-2012.tar"

if not voc_tar.exists():
    print("\n Скачивание Pascal VOC 2012")
    voc_url = "http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar"
    urllib.request.urlretrieve(voc_url, voc_tar)
    print("       Скачивание завершено")
else:
    print("\n Датасет уже скачан")

# Распаковываем архив
if not (data_dir / "VOCdevkit").exists():
    print("\n Распаковка датасета...")
    with tarfile.open(voc_tar, 'r') as tar:
        tar.extractall(path=data_dir)
    print("       Распаковка завершена")
else:
    print("\n Датасет уже распакован")

# Определяем пути к изображениям и маскам
voc_root = data_dir / "VOCdevkit" / "VOC2012"
images_dir = voc_root / "JPEGImages"
masks_dir = voc_root / "SegmentationObject"

print(f"\n Пути к данным:")
print(f"      Изображения: {images_dir}")
print(f"      Маски: {masks_dir}")

# Получаем список всех изображений
all_images = list(images_dir.glob("*.jpg"))
print(f"\n Всего изображений найдено: {len(all_images)}")

# Фильтруем изображения (оставляем только те, у которых есть маски)
valid_images = []
for img_path in all_images:
    mask_path = masks_dir / img_path.name.replace('.jpg', '.png')
    if mask_path.exists():
        valid_images.append(img_path)

print(f" Изображений с корректными масками: {len(valid_images)}")

# Случайным образом выбираем 100 изображений
# Фиксируем seed для воспроизводимости результатов
random.seed(42)
test_images = random.sample(valid_images, 100)

print(f"\n Случайным образом выбрано изображений: {len(test_images)}")

print("\n Примеры выбранных изображений:")
for i, img_path in enumerate(test_images[:100], 1):
    print(f"      {i}. {img_path.name}")

print("\n" + "="*70)
print("100 изображений - выбраны")
print("="*70)

ВЫПОЛНЕНИЕ: Выбор 100 случайных изображений

 Скачивание Pascal VOC 2012
       Скачивание завершено

 Распаковка датасета...


/tmp/ipykernel_6509/1704313426.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=data_dir)


       Распаковка завершена

 Пути к данным:
      Изображения: /content/pascal_voc/VOCdevkit/VOC2012/JPEGImages
      Маски: /content/pascal_voc/VOCdevkit/VOC2012/SegmentationObject

 Всего изображений найдено: 17125
 Изображений с корректными масками: 2913

 Случайным образом выбрано изображений: 100

 Примеры выбранных изображений:
      1. 2011_003182.jpg
      2. 2009_001731.jpg
      3. 2010_004432.jpg
      4. 2007_001586.jpg
      5. 2010_003231.jpg
      6. 2007_003917.jpg
      7. 2007_000491.jpg
      8. 2008_004345.jpg
      9. 2010_004219.jpg
      10. 2009_003711.jpg
      11. 2007_004830.jpg
      12. 2009_003551.jpg
      13. 2011_002447.jpg
      14. 2011_001910.jpg
      15. 2010_003250.jpg
      16. 2007_003991.jpg
      17. 2008_008343.jpg
      18. 2008_005300.jpg
      19. 2008_004551.jpg
      20. 2007_001724.jpg
      21. 2010_001752.jpg
      22. 2007_001568.jpg
      23. 2011_001400.jpg
      24. 2007_005911.jpg
      25. 2007_009684.jpg
      26. 2008_004610.

**Создание выходного файла с результатами**

In [4]:
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import time

print("ВЫПОЛНЕНИЕ: Создание выходного файла")

# Функция для вычисления IoU (Intersection over Union)
def calculate_iou(pred_mask, gt_mask, num_classes=21):
    """
    Вычисляет IoU для каждого класса

    Args:
        pred_mask: предсказанная маска (H, W)
        gt_mask: ground truth маска (H, W)
        num_classes: количество классов

    Returns:
        ious: список IoU для каждого класса
        mean_iou: средний IoU
    """
    ious = []

    for cls in range(num_classes):
        # Бинарные маски для текущего класса
        pred_cls = (pred_mask == cls)
        gt_cls = (gt_mask == cls)

        # Пересечение и объединение
        intersection = (pred_cls & gt_cls).sum().item()
        union = (pred_cls | gt_cls).sum().item()

        # Вычисляем IoU
        if union == 0:
            iou = float('nan')  # Класс отсутствует
        else:
            iou = intersection / union
        ious.append(iou)

    # Средний IoU (игнорируя nan)
    mean_iou = np.nanmean(ious)
    return ious, mean_iou

print("\n Функция calculate_iou определена")

# Функция для предсказания сегментации
def predict_segmentation(image_path, model, device, preprocess):
    """
    Делает предсказание сегментации для изображения
    """
    # Загружаем изображение
    img = Image.open(image_path).convert('RGB')
    original_size = img.size

    # Применяем преобразования
    input_tensor = preprocess(img)
    input_batch = input_tensor.unsqueeze(0).to(device)

    # Предсказание
    with torch.no_grad():
        output = model(input_batch)['out'][0]

    # Получаем предсказанный класс для каждого пикселя
    output_predictions = output.argmax(0).cpu().numpy()

    # Приводим к оригинальному размеру
    pred_mask = Image.fromarray(output_predictions.astype(np.uint8))
    pred_mask = pred_mask.resize(original_size, Image.NEAREST)

    return np.array(pred_mask)

print("Функция predict_segmentation определена")

# Обрабатываем все 100 изображений и собираем результаты
print("\n Обработка 100 изображений моделью...")

results = []  # Список для хранения результатов
all_ious = [] # Список всех mean IoU

start_time = time.time()

for idx, img_path in enumerate(tqdm(test_images, desc="Обработка")):
    # Путь к ground truth маске
    mask_path = masks_dir / img_path.name.replace('.jpg', '.png')

    # Делаем предсказание
    pred_mask = predict_segmentation(img_path, model, device, preprocess)

    # Загружаем ground truth
    gt_mask = np.array(Image.open(mask_path))

    # Приводим к одному размеру если нужно
    if pred_mask.shape != gt_mask.shape:
        pred_mask = np.array(
            Image.fromarray(pred_mask).resize(
                (gt_mask.shape[1], gt_mask.shape[0]),
                Image.NEAREST
            )
        )

    # Вычисляем IoU
    class_ious, mean_iou = calculate_iou(pred_mask, gt_mask)

    # Считаем количество объектов (классов кроме фона)
    num_objects = len(np.unique(gt_mask)) - 1
    num_objects = max(num_objects, 1)  # Минимум 1

    # Сохраняем результат
    results.append({
        'image_name': img_path.name,
        'mean_iou': mean_iou,
        'num_objects': num_objects,
        'class_ious': class_ious
    })

    all_ious.append(mean_iou)

elapsed_time = time.time() - start_time
print(f"\n       Обработка завершена за {elapsed_time:.1f} секунд")

# Вычисляем общий средний IoU
overall_mean_iou = np.mean(all_ious)
print(f"\n Общий средний IoU по всем изображениям: {overall_mean_iou:.6f}")

# Создаем выходной файл
output_file = '/content/results_of_segmentation.txt'

print(f"\n Создание файла: {output_file}")

with open(output_file, 'w', encoding='utf-8') as f:
    # СТРОКА 1: средний IoU по всем изображениям
    f.write(f"{overall_mean_iou:.6f}\n")

    # СТРОКИ 2-101: результаты для каждого изображения
    for result in results:
        # Формируем строку: <имя> <средний IoU> <кол-во объектов> [IoU по классам]
        line = f"{result['image_name']} {result['mean_iou']:.6f} {result['num_objects']}"

        # Добавляем IoU по классам в квадратных скобках
        class_ious_str = ' '.join([
            f"{iou:.4f}" if not np.isnan(iou) else "nan"
            for iou in result['class_ious']
        ])
        line += f" [{class_ious_str}]"

        f.write(line + "\n")

print(f"\n Файл успешно создан!")
print(f"      Размер файла: {os.path.getsize(output_file)} байт")
print(f"      Количество строк: 101 (1 заголовок + 100 результатов)")

# Показываем первые несколько строк файла для проверки
print(f"\n Все 100 строк файла:")
with open(output_file, 'r') as f:
    for i, line in enumerate(f):
        if i < 100:
            if len(line) > 100:
                print(f"      Строка {i+1}: {line[:100]}...")
            else:
                print(f"      Строка {i+1}: {line.strip()}")

print("\n" + "="*70)
print(" Выходной файл создан")
print("="*70)

# Скачиваем файл
from google.colab import files
print("\n Скачивание файла...")
files.download(output_file)
print("       Файл скачан")

ВЫПОЛНЕНИЕ: Создание выходного файла

 Функция calculate_iou определена
Функция predict_segmentation определена

 Обработка 100 изображений моделью...


Обработка: 100%|██████████| 100/100 [00:13<00:00,  7.35it/s]


       Обработка завершена за 13.6 секунд

 Общий средний IoU по всем изображениям: 0.267149

 Создание файла: /content/results_of_segmentation.txt

 Файл успешно создан!
      Размер файла: 12624 байт
      Количество строк: 101 (1 заголовок + 100 результатов)

 Все 100 строк файла:
      Строка 1: 0.267149
      Строка 2: 2011_003182.jpg 0.055112 2 [0.2756 0.0000 nan nan nan nan nan nan 0.0000 nan nan nan 0.0000 nan nan ...
      Строка 3: 2009_001731.jpg 0.329969 2 [0.9899 0.0000 nan nan nan nan nan nan nan nan nan nan nan nan nan nan na...
      Строка 4: 2010_004432.jpg 0.000000 1 [0.0000 0.0000 nan nan nan nan nan nan nan nan nan nan nan nan nan 0.0000...
      Строка 5: 2007_001586.jpg 0.192829 3 [0.9641 0.0000 0.0000 nan nan nan nan nan nan nan nan nan nan 0.0000 nan ...
      Строка 6: 2010_003231.jpg 0.328082 2 [0.9842 0.0000 nan nan nan nan nan 0.0000 nan nan nan nan nan nan nan nan...
      Строка 7: 2007_003917.jpg 0.298004 2 [0.8940 0.0000 nan nan nan nan nan 0.0000 nan 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

       Файл скачан
